# Python Chassis Motion Contral

In this chapter, we'll craft a Python routine for controlling the motion of a robot's chassis. This approach can be adapted to other programming languages for similar motion control tasks.

## Control Mechanism Overview

We utilize code blocks within JupyterLab to compose JSON commands. These commands are then dispatched to the microcontroller via the GPIO serial port on a Raspberry Pi (the default baud rate for communication with the microcontroller is 115200). Upon receiving these commands, the microcontroller executes the specified actions.

Further sections will delve into the variety of commands that can be sent to the microcontroller. Alternatively, you might choose to implement these functionalities in a different programming language or develop a dedicated application for the controlling system.

## Design Advantages

Adopting a master-slave architecture significantly liberates the resources of the master device. In this setup, the master device (such as a Raspberry Pi or Jetson SBC) acts as the brain, while the ESP32 microcontroller serves as a cerebellum-like entity handling the lower-level motion controls. This division of labor allows the master device to focus on high-level tasks like vision processing and decision-making, while the microcontroller efficiently manages precise movement control and other low-level tasks. Such an arrangement ensures that the microcontroller can maintain accurate wheel rotation speeds through high-frequency PID control, without overburdening the master device with computationally intensive tasks.

## Main Program (app.py)

The main program of the product, app.py, automatically executes upon booting due to the configuration set by autorun.sh (which is pre-configured in the product). Running app.py occupies the GPIO serial port and the camera, which might lead to conflicts or errors if other tutorials or programs require these resources. Ensure to disable the auto-start of app.py before proceeding with development or learning.

Since app.py uses multithreading and is set to run automatically at startup via a service, you usually cannot use commands like 'sudo killall python' to stop app.py. You need to execute the following commands and wait for them to finish; there is no need to reboot the robot product.

### Ending the Main Program
1. Click the “ ” icon next to the tab of this page at the top, which will open a new tab called Launcher.
2. Click Terminal under Other to open the terminal window.
3. Type `bash` in the terminal window and press Enter.
4. You can now use the Bash Shell to control the robot.
5. Enter the command: `systemctl --user stop ugv-app.service`.
6. After the command is executed in the terminal page, continue with the remaining parts of this tutorial.

Note: 

If you need to permanently stop the app.py program, execute systemctl --user disable ugv-app.service;

After restarting the device, the main product program will no longer run automatically on startup, and you can freely use the tutorials in JupyterLab.

If later you need to restore the main program to run automatically on startup, execute systemctl --user enable ugv-app.service, and this will restore the main program's automatic startup.


## Chassis Control Routine

In the following routine, we use the is_raspberry_pi5() function to determine the model of the Raspberry Pi since the device names for the GPIO serial port differ between Raspberry Pi 4B and Raspberry Pi 5. It's crucial to use the correct device name and baud rate (default 115200) that matches the microcontroller.

Before executing the code block below, ensure the robot is elevated so that the drive wheels are off the ground. Activating the code will cause the robot to move; take precautions to prevent it from falling off the table.

In [ ]:
from base_ctrl import BaseController
import time

# Function for Detecting Raspberry Pi
def is_raspberry_pi5():
    with open('/proc/cpuinfo', 'r') as file:
        for line in file:
            if 'Model' in line:
                if 'Raspberry Pi 5' in line:
                    return True
                else:
                    return False

# Determine the GPIO Serial Device Name Based on the Raspberry Pi Model
if is_raspberry_pi5():
    base = BaseController('/dev/ttyAMA0', 115200)
else:
    base = BaseController('/dev/serial0', 115200)

# The wheel rotates at a speed of 0.2 meters per second and stops after 2 seconds.
base.send_command({"T":1,"X":0.2,"Y":0.0,"Yaw":0.0})
time.sleep(2)
base.send_command({"T":1,"X":0.0,"Y":0.0,"Yaw":0.0})

By calling the code block above, the Raspberry Pi will first send the instruction `{"T":1,"X":0.2,"Y":0.0,"Yaw":0.0}` (we will explain the instruction structure in more detail in later chapters) to start moving. After a two-second interval, the Raspberry Pi will send the instruction `{"T":1,"X":0.0,"Y":0.0,"Yaw":0.0}` to stop. It's important to note that even without sending the subsequent stop instruction, the rotation will still stop if no new instructions are sent. This is because the lower-level machine contains a heartbeat function. The heartbeat function's purpose is to automatically stop the current movement instruction when the upper-level machine does not send new instructions for an extended period. This function is designed to prevent the lower-level machine from continuing to move if the upper-level machine crashes for some reason.

If you want the robot to move continuously, the upper-level machine needs to send motion control instructions cyclically every 2-4 seconds.

## Movement Principle

The hexapod robot's motion control employs a velocity command method based on the body coordinate system. In the control command, X and Y represent the translational velocity components in the forward and lateral directions of the robot, respectively, and Yaw represents the angular velocity of the robot about its vertical axis. When Yaw is not zero, the robot superimposes a rotational component onto the target trajectory of each leg in the world coordinate system, causing differential displacement of the supporting legs, thus forming a rotational motion around the center of the body, achieving overall turning.

During the turning process, the gait timing of each leg remains unchanged. By performing motion compensation on the supporting phase and the leg-lifting phase, stable contact between the feet and the ground is ensured, thereby achieving smooth and continuous turning motion in place or with accompanying translation.